# Laya V4.7 — FULL + Two Shared-Window Attention Layers (Novel Weight-Tied Variant)

**Open the model experiment on an NVIDIA T4 Colab runtime.**

The ModernBERT paper already uses a repeating **FULL → LOCAL(128) → LOCAL(128)** pattern. Its 28-layer encoder contains ten full layers and nine local-layer pairs. The V4.6 [vtava/Laya-1.2-speedup-standalone](https://huggingface.co/vtava/Laya-1.2-speedup-standalone) already has this pattern. *Reapplying it unmodified would not reduce model size or speed.*

**New idea in V4.7:** keep that precise alternating receptive field but tie the **QKV and output projection parameters** of each pair of adjacent local-window layers. Both local layers still execute independent attention operations, with separate layer norm, residuals and full FFNs. Full-attention layers remain untouched. Retain V4.6's 64-token marker-aware decision head.

```text
0  FULL  ── 1  LOCAL 128 ── 2  LOCAL 128   (layers 1 & 2 share attention Wqkv, Wo)
3  FULL  ── 4  LOCAL 128 ── 5  LOCAL 128   (layers 4 & 5 share attention Wqkv, Wo)
...
24 FULL  ── 25 LOCAL 128 ── 26 LOCAL 128  (layers 25 & 26 share attention Wqkv, Wo)
27 FULL
```

**Expected unique weights saved**: 9 pairs × (3d² + d²) at d=1024 = **37,748,736 parameters**, approximately **151 MB FP32**. Exact checkpoint bytes are calculated at runtime; tied follower tensor aliases are omitted from `model.safetensors`. Weight sharing **does not reduce attention FLOPs or guarantee faster inference**; ModernBERT's original local windows already save the quadratic attention cost.

This notebook:
1. Loads published full V4.6 standalone weights as the independent reference.
2. Creates a genuinely **different** paired-local-attention model with separate FFNs.
3. Runs optional teacher/student distillation updating *only* the nine shared attention projection sets.
4. Selects its checkpoint on a disjoint PROBE set; evaluates AUDIT exactly once after freezing.
5. Benchmarks GPU and wall-clock performance, real checkpoint size and numerical reload parity.
6. Saves full standalone, offline-loadable model in `/content/Laya_V47_FLL_Shared/`.
7. Uploads to **vtava/Laya-v47-full-2shared-windows** only if the accuracy, size and speed-nonregression gates pass, with your explicit Hugging Face write login. The prior models remain untouched.

**Research risk**: tying pretrained projections removes representational freedom and may significantly hurt fidelity. This is an experimental compression; *no quality or speed claims* are made until the Colab has run. Publication is blocked on failed gates.

References: [ModernBERT official architecture](https://huggingface.co/docs/transformers/v4.57.6/en/model_doc/modernbert), [ModernBERT paper](https://aclanthology.org/2025.acl-long.127/), [standalone V4.7 implementation](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/standalone/laya_fll_shared/modeling_laya_fll_shared.py).


In [ ]:
#@title 1. Install pinned libraries (GPU required for paired benchmarking)
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json, random, time, gc, math, importlib.util, copy, statistics, hashlib
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download, notebook_login
from safetensors.torch import save_file, load_file
assert torch.cuda.is_available(), "Please select a Colab GPU (T4 or better)"
print("GPU:",torch.cuda.get_device_name(0),"Torch:",torch.__version__)


In [ ]:
#@title 2. Experiment configuration and download complete V4.6 standalone checkpoint
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
STUDENT_REPO="vtava/Laya-v47-full-2shared-windows"
SEED=2047
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=1 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224 #@param {type:"integer"}
TRAIN_STEPS=120 #@param {type:"integer"}
TRAIN_CHECK_EVERY=30 #@param {type:"integer"}
TRAIN_LR=0.000008 #@param {type:"number"}
SHARE_INIT="first" #@param ["first","average"]
PROBE_DECISIONS=120
AUDIT_DECISIONS=200
BENCH_REPEATS=12 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.99 #@param {type:"number"}
MAX_GOLD_DROP_PP=1.5 #@param {type:"number"}
MAX_JS=0.003
MIN_SPEED_RATIO=0.95 #@param {type:"number"}
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert TRAIN_STEPS>=0 and TRAIN_CHECK_EVERY>0 and TRAIN_MAX_TOKENS>=128
assert SHARE_INIT in ("first","average")
assert 0<MIN_SPEED_RATIO<=1.0
OUT=Path("/content/Laya_V47_FLL_Shared")
OUT.mkdir(exist_ok=True,parents=True)
DEST=OUT/"standalone_model"
DEST.mkdir(exist_ok=True,parents=True)
device=torch.device("cuda")
amp_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)

api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO)
except Exception as exc:
    print("Source may require Hugging Face login:",repr(exc))
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO)
parent_dir=Path(snapshot_download(TEACHER_REPO))
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert source_config["head_context_budget"]==64
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
assert tokenizer.pad_token_id is not None
print("Teacher:",TEACHER_REPO,"revision",teacher_info.sha)
print("Original full checkpoint bytes:",(parent_dir/"model.safetensors").stat().st_size)


In [ ]:
#@title 3. Load exact standalone model and V4.7 code (embedded, no unpinned imports)
# This code is stored here and later in the new model repo. No remote custom
# transformers code/trust_remote_code at inference.
STUDENT_RUNTIME_CODE = "\"\"\"Laya V4.7 FULL → local → local, tied window-attention weights.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\n\n# ---------------------------------------------------------------------\n# V4.7: every FULL attention block is followed by 2 equal local windows.\n# The two local layers TIE Wqkv and Wo parameters. Their layer norms, MLPs,\n# residuals, RoPE, and layer-specific attention output are NOT shared.\n# ---------------------------------------------------------------------\ndef local_pair_indices(model):\n    layers=model.encoder.layers\n    n=len(layers)\n    result=[]\n    # ModernBERT is already global/local/local. Verify every selected pair.\n    for anchor in range(0,n,3):\n        pair=(anchor+1,anchor+2)\n        if pair[1]>=n:\n            continue\n        full=layers[anchor].attn\n        left=layers[pair[0]].attn\n        right=layers[pair[1]].attn\n        assert full.local_attention==(-1,-1),(\n            f\"Layer {anchor} must be full attention\")\n        assert left.local_attention==right.local_attention,(\n            f\"Local layers {pair} must use same attention window\")\n        assert left.local_attention!=(-1,-1),(\n            f\"Local layers {pair} must not be full attention\")\n        result.append(pair)\n    return result\n\n\ndef tie_paired_attention(model, *, initialization=\"first\"):\n    \"\"\"Share only attention projection parameters within paired local layers.\n\n    initialization='first' copies the first local attention's pretrained\n    projection into its follower by aliasing.\n    initialization='average' averages pretrained tensors before aliasing.\n    This loses the other original tensor: quality is NOT guaranteed, and\n    student/teacher distillation and an independent holdout are essential.\n    \"\"\"\n    if initialization not in (\"first\",\"average\"):\n        raise ValueError(\"initialization must be first or average\")\n    pairs=local_pair_indices(model)\n    dropped=[]\n    with torch.no_grad():\n        for leader_idx,follower_idx in pairs:\n            a=model.encoder.layers[leader_idx].attn\n            b=model.encoder.layers[follower_idx].attn\n            for component in (\"Wqkv\",\"Wo\"):\n                leader=getattr(a,component)\n                follower=getattr(b,component)\n                if leader.weight.shape!=follower.weight.shape:\n                    raise ValueError(f\"Incompatible projections {component}\")\n                if initialization==\"average\":\n                    leader.weight.data.add_(follower.weight.data).mul_(0.5)\n                follower.weight=leader.weight\n                dropped.append(\n                    f\"encoder.layers.{follower_idx}.attn.{component}.weight\")\n                if leader.bias is not None or follower.bias is not None:\n                    if leader.bias is None or follower.bias is None:\n                        raise ValueError(\"Asymmetric projection bias\")\n                    if initialization==\"average\":\n                        leader.bias.data.add_(follower.bias.data).mul_(0.5)\n                    follower.bias=leader.bias\n                    dropped.append(\n                        f\"encoder.layers.{follower_idx}.attn.{component}.bias\")\n    return pairs,dropped\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef export_tied_state(model):\n    \"\"\"One independent tensor per unique attention parameter.\"\"\"\n    expected=set()\n    for left,right in local_pair_indices(model):\n        a=model.encoder.layers[left].attn\n        b=model.encoder.layers[right].attn\n        for component in (\"Wqkv\",\"Wo\"):\n            aa=getattr(a,component); bb=getattr(b,component)\n            if aa.weight is not bb.weight:\n                raise AssertionError(f\"Pair {left},{right} does not share {component}.weight\")\n            expected.add(f\"encoder.layers.{right}.attn.{component}.weight\")\n            if aa.bias is not None:\n                if aa.bias is not bb.bias:\n                    raise AssertionError(f\"Pair {left},{right} does not share {component}.bias\")\n                expected.add(f\"encoder.layers.{right}.attn.{component}.bias\")\n    state=model.state_dict()\n    assert expected.issubset(state.keys())\n    # Tied aliases are omitted, not serialized twice.\n    packed={k:v.detach().cpu().clone().contiguous()\n            for k,v in state.items() if k not in expected}\n    return packed,sorted(expected)\n\n\ndef load_tied_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Offline inference loader; all trained weights live in one checkpoint.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-fll-shared-v1\"\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    pairs,dropped=tie_paired_attention(model,initialization=\"first\")\n    assert [list(p) for p in pairs]==metadata[\"shared_attention_pairs\"]\n    assert sorted(dropped)==sorted(metadata[\"omitted_alias_keys\"])\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    # missing keys MUST be exactly the intentionally tied followers.\n    res=model.load_state_dict(weights,strict=False)\n    assert not res.unexpected_keys,res.unexpected_keys\n    assert sorted(res.missing_keys)==sorted(dropped),(res.missing_keys,dropped)\n    # The leader param and follower alias are exactly the same object.\n    for a,b in pairs:\n        la=model.encoder.layers[a].attn\n        lb=model.encoder.layers[b].attn\n        assert la.Wqkv.weight is lb.Wqkv.weight\n        assert la.Wo.weight is lb.Wo.weight\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n"
runtime_file=OUT/"modeling_laya_fll_shared.py"
runtime_file.write_text(STUDENT_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("laya_v47_shared",str(runtime_file))
assert spec and spec.loader
v47=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v47)

teacher,teacher_tokenizer=v47.load_model(parent_dir,device="cuda",
                                        local_files_only=True,compile_head=False)
teacher.eval().requires_grad_(False)
student,_=v47.load_model(parent_dir,device="cuda",
                         local_files_only=True,compile_head=False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
unique_native=v47.unique_parameter_count(teacher)
pairs,omitted=v47.tie_paired_attention(student,initialization=SHARE_INIT)
unique_student=v47.unique_parameter_count(student)
saved_unique=unique_native-unique_student
assert len(pairs)==9,pairs
assert saved_unique>30_000_000,saved_unique
assert all(student.encoder.layers[a].attn.Wqkv.weight is
           student.encoder.layers[b].attn.Wqkv.weight for a,b in pairs)
for a,b in pairs:
    assert student.encoder.layers[a].mlp.Wi.weight is not student.encoder.layers[b].mlp.Wi.weight
print("Shared local-attention pairs:",pairs)
print("Unique parameters before:",f"{unique_native:,}")
print("Unique parameters after: ",f"{unique_student:,}")
print("Unique parameters removed:",f"{saved_unique:,}",
      "FP32 byte estimate",f"{saved_unique*4:,}")
print("Original FFNs are kept distinct; full attention remains unchanged.")


In [ ]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

assert len(probe_items)>=PROBE_DECISIONS and len(audit_items)>=AUDIT_DECISIONS
assert {x["case_id"] for x in probe_items}.isdisjoint({x["case_id"] for x in audit_items})
print("TRAIN/PROBE/AUDIT items:",len(train_items),len(probe_items),len(audit_items))


In [ ]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_SPEED_RATIO)



In [ ]:
#@title 6. PROBE-only benchmark and original-size baseline
PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
before_probe=quality(student,probe_items,PROBE_REF)
original_size=(parent_dir/"model.safetensors").stat().st_size
# The real difference is determined by safetensors dedup (and includes headers).
packed_initial,drop_keys=v47.export_tied_state(student)
estimated_tensor_bytes=sum(v.numel()*v.element_size() for v in packed_initial.values())
del packed_initial
print("Initial shared-weights PROBE:",before_probe)
print("Unique native parameters:",unique_native)
print("Unique shared parameters:",unique_student)
print("Omitted tied tensor alias keys:",len(drop_keys))
print("Estimated deduplicated tensor bytes:",estimated_tensor_bytes)
print("Native checkpoint bytes:",original_size)


In [ ]:
#@title 7. Distill teacher decisions into only the paired LOCAL attention matrices
# Shared attention training leaves all 28 FFNs, FULL layers, norms, output
# scorer, type embeddings and V4.6 decision head unchanged.
shared_parameters=[]
for a,b in pairs:
    attn=student.encoder.layers[a].attn
    for linear in (attn.Wqkv,attn.Wo):
        shared_parameters.append(linear.weight)
        if linear.bias is not None:
            shared_parameters.append(linear.bias)
ids=[id(p) for p in shared_parameters]
assert len(set(ids))==len(ids)
for p in shared_parameters:p.requires_grad_(True)
assert sum(p.numel() for p in student.parameters() if p.requires_grad)==saved_unique
# Evaluate in eval mode for deterministic dropout-free distillation; grads work.
student.eval()
optimizer=torch.optim.AdamW(shared_parameters,lr=TRAIN_LR,weight_decay=0)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
def shared_snapshot():
    return [p.detach().cpu().clone() for p in shared_parameters]
def restore_shared(saved):
    with torch.no_grad():
        for p,tensor in zip(shared_parameters,saved):
            p.copy_(tensor.to(p.device))
selection=[]
best_state=shared_snapshot()
best_probe=before_probe
best_step=0
best_feasible=valid_quality(before_probe)

def train_batch_at(step):
    # All option-marker tokens lie in the question header <=192; truncate only
    # the trailing state to keep T4 backprop memory manageable.
    items=[train_items[((step-1)*TRAIN_B+i)%len(train_items)] for i in range(TRAIN_B)]
    b=direct_batch(items)
    for key in ("input_ids","attention_mask"):
        b[key]=b[key][:,:TRAIN_MAX_TOKENS]
    assert (b["marker_pos"][b["marker_mask"]]<TRAIN_MAX_TOKENS).all(),(
        "Option marker truncated by TRAIN_MAX_TOKENS; increase it")
    return b

for step in range(1,TRAIN_STEPS+1):
    b=train_batch_at(step)
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**b)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        student_logits,_=student(**b)
    temp=2.0
    target=F.softmax(teacher_logits.float()/temp,dim=-1)
    log_p=F.log_softmax(student_logits.float()/temp,dim=-1)
    loss=F.kl_div(log_p,target,reduction="batchmean")*temp**2
    assert torch.isfinite(loss),("Non-finite distillation loss",step)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(shared_parameters,max_norm=1.)
    scaler.step(optimizer)
    scaler.update()
    if step%TRAIN_CHECK_EVERY==0 or step==TRAIN_STEPS:
        current=quality(student,probe_items,PROBE_REF)
        feasible=valid_quality(current)
        improve=(feasible and (not best_feasible or
                 current["mean_js"]<best_probe["mean_js"]))
        # If no feasible checkpoint exists, retain the lowest-JS candidate
        # for a transparent *failed* experimental archive (never publish).
        if not best_feasible and current["mean_js"]<best_probe["mean_js"]:
            improve=True
        if improve:
            best_step=step
            best_state=shared_snapshot()
            best_probe=current
            best_feasible=feasible
        row=dict(step=step,loss=float(loss.detach().cpu()),quality=current,
                 meets_probe_quality_gates=bool(feasible),selected=bool(improve))
        selection.append(row)
        (OUT/"training_progress.json").write_text(json.dumps(selection,indent=2))
        print("PROBE step",step,"JS",current["mean_js"],
              "agreement",current["teacher_agreement"],
              "gold",current["gold_accuracy"],"valid",feasible,
              "selected",improve)
restore_shared(best_state)
student.eval().requires_grad_(False)
print("Frozen selected checkpoint: step",best_step,
      "PROBE valid",best_feasible)
# No AUDIT samples have been used yet.


In [ ]:
#@title 8. Frozen PROBE benchmark, then exactly ONE untouched AUDIT
frozen_probe=quality(student,probe_items,PROBE_REF)
frozen_speed=benchmark_paired(teacher,student,repeats=BENCH_REPEATS)
print("PROBE after distillation:",frozen_probe)
print("Paired GPU speedup:",frozen_speed["median_gpu_speedup"])
print("Minimum full-model GPU speed ratio:",frozen_speed["min_gpu_speedup"])
print("Minimum wall-clock speed ratio:",frozen_speed["min_wall_speedup"])
# Do not touch the student/architecture after this point.
AUDIT_REF=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in AUDIT_REF}.isdisjoint({r["case"] for r in PROBE_REF})
final_quality=quality(student,audit_items,AUDIT_REF)
final_speed=benchmark_paired(teacher,student,repeats=max(15,BENCH_REPEATS))
print("Untouched AUDIT:",final_quality)
print("Final speed:",final_speed)


In [ ]:
#@title 9. Save full standalone checkpoint with tied weights deduplicated
from collections import OrderedDict
DEST.mkdir(exist_ok=True,parents=True)
(DEST/"encoder").mkdir(exist_ok=True)
(DEST/"tokenizer").mkdir(exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_fll_shared.py").write_text(STUDENT_RUNTIME_CODE,encoding="utf-8")
packed,omitted_actual=v47.export_tied_state(student)
assert omitted_actual==sorted(omitted),("Tied key tracking mismatch",omitted_actual,omitted)
model_path=DEST/"model.safetensors"
save_file(packed,str(model_path),metadata={
    "format":"pt",
    "architecture":"laya-fll-shared-v1",
    "teacher_revision":str(teacher_info.sha),
})
del packed
gc.collect()
compressed_size=model_path.stat().st_size
actual_saved_bytes=original_size-compressed_size
assert compressed_size<original_size,("Tied model failed to shrink checkpoint",
                                       original_size,compressed_size)
cfg={
 "format":"laya-fll-shared-v1",
 "model_type":"laya-v47-alternating-shared-local",
 "head_context_budget":64,
 "head_layers":int(source_config["head_layers"]),
 "source_model":TEACHER_REPO,
 "source_revision":teacher_info.sha,
 "shared_attention_pairs":[list(p) for p in pairs],
 "omitted_alias_keys":sorted(omitted),
 "original_full_attn_period":3,
 "local_attention_window":128,
 "initialization":SHARE_INIT,
 "train_steps_requested":TRAIN_STEPS,
 "trained_checkpoint_selected_at_step":best_step,
 "model_scope":"option-score logits only, no auxiliary action head",
 "compression_type":"tied adjacent local attention QKV and output projections; all FFNs independent",
}
(DEST/"config.json").write_text(json.dumps(cfg,indent=2))
# Confirm model can be constructed strictly from saved local files; no parents.
reload_model,reload_tokenizer=v47.load_tied_model(DEST,device="cuda")
b=batch(probe_items,0,BATCH)
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    original_logits,_=student(**b)
    reloaded_logits,_=reload_model(**b)
parity_max_diff=float((original_logits.float()-reloaded_logits.float()).abs().max())
parity_same=bool(torch.equal(original_logits.argmax(-1),reloaded_logits.argmax(-1)))
parity_close=bool(torch.allclose(original_logits.float(),reloaded_logits.float(),
                                 atol=2e-3,rtol=2e-3))
assert parity_same and parity_close,(parity_max_diff,parity_same,parity_close)
del reload_model
gc.collect();torch.cuda.empty_cache()
print("Local-only reload parity PASS; max absolute logit diff:",parity_max_diff)
print("Unique attention parameters removed:",f"{saved_unique:,}")
print("Model checkpoint smaller by:",f"{actual_saved_bytes:,} bytes",
      f"({100*actual_saved_bytes/original_size:.2f}%)")


In [ ]:
#@title 10. Final size/fidelity gates, model card and report
gates={
 "probe_quality":valid_quality(frozen_probe),
 "holdout_teacher_agreement":final_quality["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
 "holdout_gold_drop":final_quality["gold_drop"]<=MAX_GOLD_DROP_PP/100.,
 "holdout_distribution_js":final_quality["mean_js"]<=MAX_JS,
 "gpu_no_significant_regression":final_speed["min_gpu_speedup"]>=MIN_SPEED_RATIO,
 "wall_no_significant_regression":final_speed["min_wall_speedup"]>=MIN_SPEED_RATIO,
 "size_saving_at_least_100MB":actual_saved_bytes>=100_000_000,
 "offline_reload_parity":parity_same and parity_close,
}
gates["all_pass"]=all(gates.values())
def total_bytes(d):return sum(p.stat().st_size for p in d.rglob("*") if p.is_file())
final_report={
 "version":"V4.7",
 "architecture":"FULL / LOCAL(128) / LOCAL(128), local projection weights shared in nine pairs",
 "source_model":TEACHER_REPO,
 "source_revision":teacher_info.sha,
 "gpu":torch.cuda.get_device_name(0),
 "native_parameters":unique_native,"shared_parameters":unique_student,
 "removed_unique_parameters":saved_unique,
 "native_checkpoint_bytes":original_size,
 "shared_checkpoint_bytes":compressed_size,
 "saved_checkpoint_bytes":actual_saved_bytes,
 "saved_checkpoint_fraction":actual_saved_bytes/original_size,
 "initialization":SHARE_INIT,
 "requested_steps":TRAIN_STEPS,
 "selected_training_step":best_step,
 "probe_selection":selection,
 "final_probe_quality":frozen_probe,
 "final_probe_speed":frozen_speed,
 "final_audit":final_quality,
 "final_speed":final_speed,
 "offline_reload_max_abs_diff":parity_max_diff,
 "gates":gates,
 "disclaimer":"Weight sharing reduces unique weight storage but not mathematical attention FLOPs. Local attention compute savings were present in original ModernBERT."
}
(DEST/"v47_report.json").write_text(json.dumps(final_report,indent=2))
card=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- attention-weight-sharing
- local-global-attention
- research
- standalone
---

# Laya V4.7 — FULL + 2 shared local windows

28-layer pretrained ModernBERT encoder; original repeating
`FULL / LOCAL(128) / LOCAL(128)` pattern. We tie only the attention QKV and
output weights of each pair of adjacent local layers while **keeping all
FFNs, norms, residuals, global layers and V4.6 64-token decision head
independent and intact**.

The original ModernBERT already used FULL + 2 LOCAL. The new contribution
is the **pairwise attention parameter sharing**, which saves distinct weights
but does **not** decrease attention FLOPs by itself.

- Original unique weights: **{unique_native:,}**
- New unique weights: **{unique_student:,}**
- Unique parameters removed: **{saved_unique:,}**
- Original checkpoint bytes: **{original_size:,}**
- New compressed checkpoint bytes: **{compressed_size:,}**
- Disk saving: **{actual_saved_bytes:,} bytes**
- Audit agreement: **{final_quality['teacher_agreement']:.3%}**
- Gold accuracy: **{final_quality['gold_accuracy']:.3%}**
- Teacher gold accuracy: **{final_quality['teacher_gold_accuracy']:.3%}**
- Mean JS divergence: **{final_quality['mean_js']:.7f}**
- Paired GPU speed ratio: **{final_speed['median_gpu_speedup']:.3f}×**
- All publication gates passed: **{gates['all_pass']}**

## Offline inference

All weights, encoder config, tokenizer and the inference implementation
are included in this repository. After downloading it once:

```python
from modeling_laya_fll_shared import load_tied_model
model,tokenizer=load_tied_model("/path/to/downloaded/model",device="cuda")
answers=model.predict(tokenizer,
    dict(road_direction="left",speed=7),
    dict(steering=dict(type="choice",instructions="Select direction",
      criteria=dict(left="left curve",straight="straight road",right="right curve"))))
print(answers)
```

Download the model repo from Hugging Face using `snapshot_download`;
after that, no external model download is needed.

This is an **experimental distillation/compression** of an existing
pretrained model, not evidence that tied weights are universally faster
or that a different attention pattern was invented. The source
ModernBERT alternating window design predates this experiment.
"""
(DEST/"README.md").write_text(card)
(DEST/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
(DEST/"v47_report.json").write_text(json.dumps({
    **final_report,"total_model_repo_files_bytes":total_bytes(DEST)},indent=2))
print("PUBLICATION GATES:",json.dumps(gates,indent=2))
print("FULL CHECKPOINT:",model_path)
print("MODEL FILES BYTES:",total_bytes(DEST))
print("V4.7 report:",DEST/"v47_report.json")
if not gates["all_pass"]:
    print("EXPERIMENT FAILED VALIDATION: new model files were saved locally, but do NOT publish as successful variant.")
else:
    print("PASS: new attention-tied architecture validated on this benchmark; independent testing still recommended.")


In [ ]:
#@title 11. Publish ONLY if validated; private separate Hugging Face model repo
if not gates["all_pass"]:
    print("Upload BLOCKED because one or more quality/size/speed gates failed.")
elif PUBLISH_TO_HF:
    notebook_login()
    uploader=HfApi()
    account=uploader.whoami()["name"]
    assert account=="vtava",f"Expected vtava account; got {account}"
    assert STUDENT_REPO=="vtava/Laya-v47-full-2shared-windows"
    if uploader.repo_exists(STUDENT_REPO,repo_type="model"):
        names=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
        if names-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError(
                f"{STUDENT_REPO} already contains files; refusing overwrite.")
    else:
        created=uploader.create_repo(STUDENT_REPO,repo_type="model",
                                     private=PRIVATE_REPO,exist_ok=False)
        print("Created:",str(created))
    uploader.upload_large_folder(
        repo_id=STUDENT_REPO,repo_type="model",folder_path=str(DEST))
    files=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
    required={"model.safetensors","modeling_laya_fll_shared.py","config.json",
              "encoder/config.json","v47_report.json","README.md"}
    assert required.issubset(files),required-files
    print("PUBLISHED AND FILE-LIST VERIFIED:",
          "https://huggingface.co/"+STUDENT_REPO)
else:
    print("Publish disabled; validated model remains in:",DEST)
print("Original model preserved:",TEACHER_REPO)
